In [39]:
import numpy as np
import oyaml as yaml
import pandas as pd

from prismadv.data_models import ConstraintsWithSources
from prismadv.data_models.config import PrismaDVConfig
from prismadv.data_models.validated_results import ValidationResults
from prismadv.project_manager.manager.base import ProjectManager
from prismadv.utils import get_project_root

In [49]:
dataset_names = ["students", "hr_analytics", "sleep_health", "imdb", "IPL_win_prediction"]
# dataset_names = ["imdb"]
model_order = ["gemini-2.5-flash", "gpt-4.1", "gpt-4o", "gemini-2.5-pro", "gpt-5-mini", "gpt-5"]

In [63]:
results_df = pd.DataFrame(columns=[
    "llm_name", "llm_temperature", "use_async", "dataset_name", "subtask_name", "timestamp",
    "processed_data_label", "script_name", "is_clean", "constraints_on_single_column_only",
    "num_passed_warning", "num_failed_warning", "num_failed_error", "num_passed_error",
    "total_constraints",
    "predicted_as_safe", "is_safe"
])
for dataset_name in dataset_names:
    project_manager = ProjectManager(project_root=get_project_root(), dataset_name=dataset_name)
    subtask_names = project_manager.get_available_subtasks()
    for subtask_name in subtask_names:
        processed_data_labels = project_manager.get_available_processed_data_labels_for_subtask(subtask_name)
        script_path_list = project_manager.get_available_script_path_list_for_subtask(subtask_name)
        script_names = [script_path.stem for script_path in script_path_list]
        for processed_data_label in processed_data_labels:
            if int(processed_data_label) == 0:
                continue
            for script_name in script_names:
                constraints_path = project_manager.get_constraints_path(
                    subtask_name, processed_data_label, script_name)
                constraint_validation_results_dir = project_manager.get_constraints_validation_path(
                    subtask_name, processed_data_label, script_name)
                constraint_file_list = list(constraints_path.glob("post_processed_prismadv*.yaml"))
                for constraint_file in constraint_file_list:
                    timestamp = constraint_file.stem.split("--")[-1]
                    with open(f"{constraint_file}", "r") as f:
                        raw_constraint_dict = yaml.load(f, Loader=yaml.FullLoader)
                    try:
                        prismadv_config = PrismaDVConfig.from_dict(raw_constraint_dict["prismadv_config"])
                    except Exception:
                        continue
                    llm_name = prismadv_config.llm.model_name
                    llm_temperature = prismadv_config.llm.temperature
                    use_async = prismadv_config.model.use_async
                    constraints = ConstraintsWithSources.from_dict({"constraints": raw_constraint_dict["constraints"]})
                    for is_clean in [True, False]:
                        if is_clean:
                            constraint_validation_result_path = constraint_validation_results_dir / f"validation_results_on_clean_test_data__{constraint_file.stem}.yaml"
                        else:
                            constraint_validation_result_path = constraint_validation_results_dir / f"validation_results_on_corrupted_test_data__{constraint_file.stem}.yaml"
                        try:
                            validation_results = ValidationResults.from_yaml(constraint_validation_result_path)
                        except FileNotFoundError:
                            continue
                        for constraints_on_single_column_only in [True, False]:
                            num_passed_warning, num_failed_warning, num_failed_error, num_passed_error, num_non_compilable = validation_results.check_result(
                                constraints_on_single_column_only=constraints_on_single_column_only,
                            )
                            total_constraints = num_passed_warning + num_failed_warning + num_failed_error + num_passed_error
                            predicted_as_safe = (num_failed_error == 0)

                            execution_result_path = project_manager.get_execution_output_validation_path(
                                subtask_name, processed_data_label, script_name
                            ) / f"basic_metrics_evaluation.json"
                            try:
                                with open(execution_result_path, "r") as f:
                                    execution_results = yaml.load(f, Loader=yaml.FullLoader)
                            except FileNotFoundError:
                                continue
                            if is_clean == True:
                                is_safe = execution_results['clean_data_is_safe']
                            else:
                                is_safe = execution_results['corrupted_data_is_safe']
                            if llm_name not in model_order:
                                continue
                            new_row = {
                                "llm_name": llm_name,
                                "timestamp": timestamp,
                                "llm_temperature": llm_temperature,
                                "use_async": use_async,
                                "dataset_name": dataset_name,
                                "subtask_name": subtask_name,
                                "processed_data_label": processed_data_label,
                                "script_name": script_name,
                                "is_clean": is_clean,
                                "constraints_on_single_column_only": constraints_on_single_column_only,
                                "num_passed_warning": num_passed_warning,
                                "num_failed_warning": num_failed_warning,
                                "num_failed_error": num_failed_error,
                                "num_passed_error": num_passed_error,
                                "total_constraints": total_constraints,
                                "predicted_as_safe": predicted_as_safe,
                                "is_safe": is_safe
                            }
                            results_df = pd.concat([results_df, pd.DataFrame([new_row])], ignore_index=True)

/var/folders/19/8dpm1qss7mn34db1f49thmww0000gn/T/ipykernel_16486/1959646564.py:85: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  results_df = pd.concat([results_df, pd.DataFrame([new_row])], ignore_index=True)


In [64]:
unique_subset_cols = [
    "llm_name", "llm_temperature", "use_async",
    "dataset_name", "subtask_name",
    "processed_data_label", "script_name", "is_clean", "constraints_on_single_column_only"
]

df_before_drop = results_df.copy()
df_before_drop['timestamp'] = pd.to_numeric(df_before_drop['timestamp'], errors='coerce')

dedup_list = []

for ds in df_before_drop['dataset_name'].unique():
    df_ds = df_before_drop[df_before_drop['dataset_name'] == ds].copy()
    df_ds = df_ds.sort_values('timestamp')
    df_unique_ds = df_ds.drop_duplicates(subset=unique_subset_cols, keep='first')
    dedup_list.append(df_unique_ds)

# Final deduplicated DataFrame across all datasets
df_all_unique = pd.concat(dedup_list, ignore_index=True)

print("Final deduplicated shape:", df_all_unique.shape)

Final deduplicated shape: (6620, 17)


In [65]:
unique_subset_cols = [
    "llm_temperature", "use_async",
    "dataset_name", "subtask_name",
    "processed_data_label", "script_name", "is_clean", "constraints_on_single_column_only"
]

df_before_drop = results_df[results_df['is_safe'] == True].copy()
df_before_drop['timestamp'] = pd.to_numeric(df_before_drop['timestamp'], errors='coerce')

# Deduplicate once globally
df_unique = df_before_drop.drop_duplicates(subset=unique_subset_cols + ["llm_name"], keep="first")

for ds in df_unique['dataset_name'].unique():
    df_ds = df_unique[df_unique['dataset_name'] == ds]

    keys_gpt4o = df_ds[df_ds["llm_name"] == "gpt-4o"][unique_subset_cols].drop_duplicates()
    keys_gpt5  = df_ds[df_ds["llm_name"] == "gpt-5"][unique_subset_cols].drop_duplicates()

    only_in_gpt4o = (
        keys_gpt4o.merge(keys_gpt5, on=unique_subset_cols, how="left", indicator=True)
        .query("_merge == 'left_only'")
        .drop(columns=["_merge"])
    )

    print(f"=== Dataset: {ds} ===")
    print("Combinations present in gpt-4o but not in gpt-5:", only_in_gpt4o.shape[0])
    if not only_in_gpt4o.empty:
        print(only_in_gpt4o.to_string(index=False))
    print()

=== Dataset: students ===
Combinations present in gpt-4o but not in gpt-5: 0

=== Dataset: hr_analytics ===
Combinations present in gpt-4o but not in gpt-5: 0

=== Dataset: sleep_health ===
Combinations present in gpt-4o but not in gpt-5: 0

=== Dataset: imdb ===
Combinations present in gpt-4o but not in gpt-5: 2
 llm_temperature use_async dataset_name subtask_name processed_data_label     script_name is_clean constraints_on_single_column_only
             0.6      True         imdb general_task                    8 general_task_19    False                              True
             0.6      True         imdb general_task                    8 general_task_19    False                             False

=== Dataset: IPL_win_prediction ===
Combinations present in gpt-4o but not in gpt-5: 0

